# Bidirectional Go2 recoverability expansion
Python 3.11. The default cells run frozen offline validation only. The original policies remain frozen; balance smoke/pilot are separately opt-in. Transition PPO is unavailable without a validated bridge. Upload `bidirectional_recoverability_molab_20261112_v1.zip` beside this notebook.

In [ ]:
from pathlib import Path
import os,sys,json,hashlib,zipfile,subprocess
assert sys.version_info[:2]==(3,11)
root=Path("bidirectional_go2_workspace")
assert not root.exists(), "Use a fresh workspace"
with zipfile.ZipFile("bidirectional_recoverability_molab_20261112_v1.zip") as z:
    z.extractall(root)
os.chdir(root)
for name,digest in json.loads(Path("bidirectional_bundle_manifest.json").read_text()).items():
    assert hashlib.sha256(Path(name).read_bytes()).hexdigest()==digest,name
subprocess.run([sys.executable,"-m","pip","install","-r","bidirectional_requirements.txt"],check=True)
os.environ.update(MUJOCO_GL="egl",WANDB_MODE="disabled",MPLCONFIGDIR="/tmp/go2_bidir_mpl",XDG_CACHE_HOME="/tmp/go2_bidir_cache")
def run(*args):
    subprocess.run([sys.executable,"-m",*args],check=True)


In [ ]:
run("pytest","-q","experiments/recoverability/test_core.py","experiments/recoverability/test_transition.py","experiments/recoverability/bidirectional/test_bidirectional.py")
run("experiments.recoverability.bidirectional.verify_models","--output","output/molab_bidir_models")


In [ ]:
run("experiments.recoverability.bidirectional.run","all","--output","output/molab_bidir_validation","--seeds-per-split","2")
run("experiments.recoverability.bidirectional.rate_coverage","--output","output/molab_bidir_rates","--budget","48")


In [ ]:
gate=json.loads(Path("output/molab_bidir_validation/validation_gate.json").read_text())
if gate["fine_tuning_eligible"]:
    run("experiments.recoverability.bidirectional.fine_tune","prepare","--validation","output/molab_bidir_validation","--output","output/molab_bidir_balance_prepared")
print(gate)


The local two-update balance smoke regressed and its ten-update pilot gate is closed. The following opt-in cell can reproduce that smoke in a fresh output directory. It never trains a transition policy.

In [ ]:
RUN_BALANCE_SMOKE=False
if RUN_BALANCE_SMOKE:
    run("experiments.recoverability.bidirectional.fine_tune","smoke","--validation","output/molab_bidir_validation","--output","output/molab_bidir_balance_smoke")


In [ ]:
RUN_BALANCE_PILOT=False
if RUN_BALANCE_PILOT:
    gate=Path("output/molab_bidir_balance_smoke/smoke_gate.json")
    assert json.loads(gate.read_text())["pilot_eligible"], "Two-update smoke rejected ten-update pilot"
    run("experiments.recoverability.bidirectional.fine_tune","pilot","--validation","output/molab_bidir_validation","--smoke-gate",str(gate),"--output","output/molab_bidir_balance_pilot")


In [ ]:
run("experiments.recoverability.bidirectional.report","--results","output/molab_bidir_validation","--rates","output/molab_bidir_rates")
print(Path("output/molab_bidir_validation/report/measured_report.md").read_text())
